# eRayz COD ULTRA — Best-of-Class CoD/Warzone YOLO Modell
---
**Verbesserungen vs. MEGA:**
- yolov8m base (besser als yolov8n)
- 9 Datasets statt 5 (~30k+ Bilder)
- AMP Mixed-Precision (2× schneller)
- Auto-Save zu Google Drive (kein Verlust bei Disconnect)
- Auto-Resume aus letztem Checkpoint
- ONNX export opset=12 (DML/AMD/Intel kompatibel)
- Pre-Training mit sunxds_0.7.8 (30k FPS images vorab)

**ANLEITUNG:**
1. Laufzeit → Laufzeittyp ändern → GPU (T4)
2. Google Drive mounten lassen (Schritt 2)
3. sunxds_0.7.8.pt hochladen (Schritt 3) — ODER direkt yolov8m.pt nutzen
4. Roboflow API Key eintragen (Schritt 4)
5. Alle Zellen ausführen
6. ~2–3h warten (Auto-Save zu Drive aktiv)
7. `erayz_cod_ultra.onnx` wird in Drive abgelegt

In [ ]:
# === SCHRITT 1: Installation ===
!pip install -q ultralytics==8.3.0 roboflow pyyaml onnx onnxruntime onnxslim
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA verfügbar: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('KEINE GPU! Laufzeit → Laufzeittyp → GPU (T4)')

In [ ]:
# === SCHRITT 2: Google Drive mounten (wichtig für Auto-Save!) ===
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/eRayz_Training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive gemountet. Output-Pfad: {DRIVE_DIR}')

In [ ]:
# === SCHRITT 3: Basis-Modell wählen ===
# Option A: yolov8m fresh (empfohlen für saubere Basis)
# Option B: sunxds_0.7.8.pt hochladen (wenn du es hast)

import os
from google.colab import files

USE_SUNXDS = False  # ← Auf True setzen, wenn du sunxds_0.7.8.pt hochladen willst

if USE_SUNXDS:
    if not os.path.exists('sunxds_0.7.8.pt'):
        print('Lade sunxds_0.7.8.pt hoch...')
        uploaded = files.upload()
    BASE_MODEL = 'sunxds_0.7.8.pt'
    print(f'Basis: sunxds_0.7.8 (Pre-trained auf 30k FPS images)')
else:
    BASE_MODEL = 'yolov8m.pt'  # wird automatisch heruntergeladen
    print(f'Basis: yolov8m (Ultralytics official)')

print(f'Verwendetes Modell: {BASE_MODEL}')

In [ ]:
# === SCHRITT 4: Datasets herunterladen (9 sorgfältig kuratiert) ===
from roboflow import Roboflow
import shutil, glob, yaml

# ═══════════════════════════════════════════════════════════════
RF_API_KEY = 'DEIN_ROBOFLOW_API_KEY_HIER'  # ← HIER eintragen!
# ═══════════════════════════════════════════════════════════════

if RF_API_KEY == 'DEIN_ROBOFLOW_API_KEY_HIER':
    raise ValueError('Bitte Roboflow API Key oben eintragen!')

rf = Roboflow(api_key=RF_API_KEY)

# 9 sorgfältig ausgewählte CoD/Warzone Datasets
datasets = [
    ('aimbots-gm16b', 'cod-complet-ubmf3', [3, 2, 1]),
    ('kolly-ku5ew', 'cod-mw-warzone', [3, 2, 1]),
    ('call-of-duty-lpxft', 'callofduty', [2, 1]),
    ('cod-auguc', 'cod-lbdrt', [1]),
    ('warzone-n5a13', 'warzone-yolov8', [2, 1]),
    ('codwarzone', 'warzone-detection', [1, 2]),
    ('aim-trainer', 'cod-mw3', [1]),
    ('jared-hampton', 'modern-warfare-detection', [1, 2]),
    ('cod-warzone-data', 'warzone-aim', [1]),
]

downloaded = []

for ws, proj, versions in datasets:
    success = False
    for v in versions:
        try:
            print(f'\n→ Lade: {proj} v{v}...')
            project = rf.workspace(ws).project(proj)
            ds = project.version(v).download('yolov8', overwrite=True)
            downloaded.append((proj, ds.location))
            success = True
            break
        except Exception as e:
            print(f'  v{v} fehlgeschlagen: {str(e)[:100]}')
            continue
    if not success:
        print(f'  ⚠ {proj} übersprungen (alle Versionen fehlgeschlagen)')

print(f'\n{"="*50}')
print(f'✓ {len(downloaded)} Datasets erfolgreich geladen!')
print(f'{"="*50}')
for name, loc in downloaded:
    n_imgs = len(glob.glob(os.path.join(loc, 'train', 'images', '*')))
    print(f'  - {name}: {n_imgs} Trainingsbilder')

In [ ]:
# === SCHRITT 5: Datasets zusammenführen → 1 Klasse 'player' ===
import re

MEGA_DIR = 'mega_dataset'
for split in ['train', 'valid']:
    os.makedirs(f'{MEGA_DIR}/{split}/images', exist_ok=True)
    os.makedirs(f'{MEGA_DIR}/{split}/labels', exist_ok=True)

img_count = 0
skipped = 0

for proj_name, ds_loc in downloaded:
    safe_name = re.sub(r'[^a-zA-Z0-9_-]', '_', proj_name)
    
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(ds_loc, split, 'images')
        lbl_dir = os.path.join(ds_loc, split, 'labels')
        
        if not os.path.exists(img_dir):
            continue
        
        # 'test' wird zu 'train' (mehr Daten = besser)
        target = 'valid' if split == 'valid' else 'train'
        
        for img_path in glob.glob(os.path.join(img_dir, '*')):
            fname = os.path.basename(img_path)
            new_fname = f'{safe_name}_{fname}'
            
            lbl_name = os.path.splitext(fname)[0] + '.txt'
            lbl_path = os.path.join(lbl_dir, lbl_name)
            
            # Skip Bilder ohne Label
            if not os.path.exists(lbl_path):
                skipped += 1
                continue
            
            # Label umschreiben: alle Klassen auf 0 (player)
            with open(lbl_path) as f:
                lines = f.readlines()
            
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = '0'
                    new_lines.append(' '.join(parts) + '\n')
            
            # Skip leere Labels
            if not new_lines:
                skipped += 1
                continue
            
            # Bild + Label kopieren
            shutil.copy2(img_path, f'{MEGA_DIR}/{target}/images/{new_fname}')
            new_lbl = os.path.splitext(new_fname)[0] + '.txt'
            with open(f'{MEGA_DIR}/{target}/labels/{new_lbl}', 'w') as f:
                f.writelines(new_lines)
            
            img_count += 1

train_count = len(glob.glob(f'{MEGA_DIR}/train/images/*'))
val_count = len(glob.glob(f'{MEGA_DIR}/valid/images/*'))

print(f'\n{"="*50}')
print(f'✓ MEGA DATASET FERTIG!')
print(f'{"="*50}')
print(f'  Training:   {train_count:,} Bilder')
print(f'  Validation: {val_count:,} Bilder')
print(f'  TOTAL:      {train_count + val_count:,} Bilder')
print(f'  Übersprungen: {skipped} (kein/leeres Label)')
print(f'  Klasse: 0 = player (alle Gegner vereint)')

In [ ]:
# === SCHRITT 6: data.yaml erstellen ===
data_yaml = {
    'path': os.path.abspath(MEGA_DIR),
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 1,
    'names': {0: 'player'}
}

yaml_path = f'{MEGA_DIR}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'✓ data.yaml geschrieben: {yaml_path}')
with open(yaml_path) as f:
    print(f.read())

In [ ]:
# === SCHRITT 7: ULTRA TRAINING ===
# Auto-Save zu Drive damit nichts verloren geht!
from ultralytics import YOLO
import shutil

model = YOLO(BASE_MODEL)
print(f'✓ Basis geladen: {BASE_MODEL}')
print(f'  Klassen original: {len(model.names)}')
print(f'  Training auf {train_count + val_count:,} CoD Bilder')
print(f'  Erwartete Dauer: 90–180 Min (T4 GPU)')
print()

# Resume-Check
RUN_NAME = 'erayz_cod_ultra'
resume_path = f'runs/detect/{RUN_NAME}/weights/last.pt'
drive_resume = f'{DRIVE_DIR}/{RUN_NAME}_last.pt'

# Wenn in Drive ein Checkpoint ist, von dort weitermachen
if os.path.exists(drive_resume) and not os.path.exists(resume_path):
    os.makedirs(os.path.dirname(resume_path), exist_ok=True)
    shutil.copy(drive_resume, resume_path)
    print(f'✓ Checkpoint aus Drive geladen: {drive_resume}')
    model = YOLO(resume_path)
    resume = True
else:
    resume = False

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,                # T4 16GB → batch 16 sicher
    device=0,
    workers=2,
    patience=15,             # Early stopping wenn 15 Epochs keine Verbesserung
    name=RUN_NAME,
    exist_ok=True,
    resume=resume,
    
    # Performance
    amp=True,                # Mixed Precision (2× schneller)
    cache='disk',            # Bilder cachen
    
    # Optimierung
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=0.0005,
    
    # CoD-spezifische Augmentation
    hsv_h=0.015,             # Farbton (geringe Variation)
    hsv_s=0.5,               # Sättigung
    hsv_v=0.4,               # Helligkeit (Tag/Nacht)
    degrees=5.0,             # Rotation (CoD ist meist nicht schief)
    translate=0.1,
    scale=0.5,               # Zoom variieren
    shear=2.0,
    perspective=0.0,         # CoD hat keine Perspektivverzerrung
    flipud=0.0,              # Spieler stehen kopf-oben
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    
    # Speichern
    save=True,
    save_period=10,          # Alle 10 Epochs Checkpoint
    plots=True,
    val=True,
)

print('\n✓ TRAINING KOMPLETT!')

In [ ]:
# === SCHRITT 8: Ergebnisse anzeigen ===
from IPython.display import Image, display

train_dir = f'runs/detect/{RUN_NAME}'
if not os.path.exists(train_dir):
    candidates = [d for d in os.listdir('runs/detect') if RUN_NAME in d]
    if candidates:
        train_dir = f'runs/detect/{sorted(candidates)[-1]}'

print(f'Training-Output: {train_dir}\n')

for img_file in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg', 'PR_curve.png']:
    fpath = f'{train_dir}/{img_file}'
    if os.path.exists(fpath):
        print(f'\n{img_file}:')
        display(Image(filename=fpath, width=900))

In [ ]:
# === SCHRITT 9: ONNX Export für Zelesis NEO (DML kompatibel) ===
best_pt = f'{train_dir}/weights/best.pt'

if not os.path.exists(best_pt):
    raise FileNotFoundError(f'best.pt nicht gefunden in {train_dir}/weights/')

best = YOLO(best_pt)

# WICHTIG: opset=12 für DirectML (AMD/Intel GPU) Kompatibilität
best.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False,
    opset=12,                # KRITISCH für Zelesis DML Backend!
    half=False,              # FP32 für maximale Zelesis-Kompatibilität
)

best_onnx = f'{train_dir}/weights/best.onnx'
out_onnx = 'erayz_cod_ultra.onnx'
out_pt = 'erayz_cod_ultra.pt'

shutil.copy(best_onnx, out_onnx)
shutil.copy(best_pt, out_pt)

# In Drive sichern
shutil.copy(out_onnx, f'{DRIVE_DIR}/{out_onnx}')
shutil.copy(out_pt, f'{DRIVE_DIR}/{out_pt}')

size_mb = os.path.getsize(out_onnx) / (1024*1024)
print(f'\n{"="*55}')
print(f'  ✓ FERTIG: erayz_cod_ultra.onnx ({size_mb:.1f} MB)')
print(f'{"="*55}')
print(f'\n  Pfad in Drive: {DRIVE_DIR}/{out_onnx}')
print(f'\n  Was im Modell:')
print(f'    • Basis: {BASE_MODEL}')
print(f'    • + {train_count + val_count:,} CoD/Warzone-spezifische Bilder')
print(f'    • + 100 Epochs Fine-Tuning')
print(f'    • → Optimiert für BO7 + Warzone')

In [ ]:
# === SCHRITT 10: Download (lokal + Drive Backup) ===
from google.colab import files

print('Lade Modelle herunter...')
print('(.onnx = für Zelesis NEO + aimbot_direct.py)')
print('(.pt = für späteres Weitertraining)')
print()
files.download(out_onnx)
files.download(out_pt)

print('\n✓ Backup auch in Google Drive:')
print(f'   {DRIVE_DIR}/{out_onnx}')
print(f'   {DRIVE_DIR}/{out_pt}')
print('\n→ Lege erayz_cod_ultra.onnx in deinen Downloads-Ordner')
print('→ Zelesis: KI-Tab → "+" → Datei wählen')
print('→ KI-Blob-Größe: 640')
print('→ Konfidenzschwelle: 0.40')